# 02 · EDA and feature set (Team 10, Saideep)

**Deliverable 3, part 2.** EDA on the team's fare file that ends in a feature set for a named
model family. Every section ends with *"…therefore, when we model, we will ___"*.

Covers: shape/types · missingness · distributions & outliers · leakage · relationships that bear
on model choice · the feature set itself (`features.py`).

Figures are saved to `team-10-modeling/visuals/` so the reports can reference them.

In [ ]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = next(p for p in [Path.cwd(), Path.cwd() / "team-10-modeling" / "notebooks"]
            if (p / "load_data.py").exists())
sys.path.insert(0, str(HERE))
VISUALS = HERE.parent / "visuals"
VISUALS.mkdir(exist_ok=True)

from load_data import CABIN_ORDER, TARGET, load_data
from features import FEATURES, build_features, clean, outlier_mask

sns.set_theme(style="whitegrid")
df = load_data()
print(df.shape)

## 1. Shape and types

In [ ]:
print(f"{len(df)} rows, {df.shape[1]} columns")
print(df.dtypes.to_string())
df.describe().T.round(2)

200 quotes and 13 columns: 1 target, 2 high-cardinality airport codes, 1 ordered category
(`cabin`), 3 binary flags and 6 numbers. No dates.

**Therefore:** with only 200 rows every extra feature costs variance, so we will engineer a
few compact features rather than one-hot encode everything, and we will validate with
repeated cross-validation rather than a single split (one 20% split is only 40 rows).

## 2. Missingness

In [ ]:
missing = df.isna().sum()
print("missing cells per column:\n", missing.to_string())
print("exact duplicate rows:", df.duplicated().sum())

No missing values and no duplicates in this file (the full synthetic sample had ~2% / ~5%
gaps in `seats_remaining` / `competitor_min_price`; those gaps do not appear here).

**Therefore:** no imputation step is needed. `clean()` still drops duplicates and missing
targets defensively, so the pipeline survives a re-cut of the data that has them.

## 3. Distribution of the target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(data=df, x=TARGET, hue="cabin", hue_order=CABIN_ORDER, bins=40,
             element="step", ax=axes[0])
axes[0].set_title("total_price by cabin (linear)")
sns.histplot(data=df, x=TARGET, hue="cabin", hue_order=CABIN_ORDER, bins=40,
             element="step", log_scale=True, ax=axes[1])
axes[1].set_title("total_price by cabin (log x-axis)")
fig.tight_layout()
fig.savefig(VISUALS / "eda_price_distribution.png", dpi=120)
plt.show()

print("skew raw:", round(df[TARGET].skew(), 2), "| skew log:", round(np.log(df[TARGET]).skew(), 2))
df.groupby("cabin", observed=True)[TARGET].agg(["count", "median", "mean", "std"]).round(0)

Price is strongly right-skewed (skew 5.7, down to 0.3 on a log scale). Much of the spread is the
cabin mix: the Business/First median is about 3× Basic Economy, and Premium Economy is about 1.5×.
Basic Economy and Main Cabin overlap almost completely in this sample (medians ~$217 vs ~$207),
so cabin is not a clean ladder at the bottom end. Distance and advance purchase do more of the work there.

**Therefore:** we will model `log(total_price)` and convert predictions back to dollars for
scoring. Cabin effects are multiplicative, which a log target turns into additive ones, so
even a linear model can represent them.

## 4. Outliers

In [ ]:
ratio = df[TARGET] / df["competitor_min_price"]
print(ratio.describe().round(3).to_string())
fig, ax = plt.subplots(figsize=(7, 5))
flag = outlier_mask(df)
ax.scatter(df.loc[~flag, "competitor_min_price"], df.loc[~flag, TARGET], s=14, label="kept")
ax.scatter(df.loc[flag, "competitor_min_price"], df.loc[flag, TARGET], s=60, c="red",
           marker="x", label="dropped (outlier rule)")
lim = [20, 4500]
ax.plot(lim, lim, "k--", lw=1, label="price = competitor")
ax.set(xscale="log", yscale="log", xlabel="competitor_min_price", ylabel="total_price",
       title="AA quote vs lowest competitor quote")
ax.legend()
fig.tight_layout()
fig.savefig(VISUALS / "eda_outliers.png", dpi=120)
plt.show()
print("rows flagged by the outlier rule:", int(flag.sum()))
df.loc[flag, ["cabin", "distance_miles", "competitor_min_price", TARGET]]

The AA quote sits between about 0.8× and 1.3× the lowest competitor quote for every row but one
(90% of rows are within 0.88×–1.13×). One
row is ~12× its competitor price (a Main Cabin fare at $3,804 on a 1,584-mile route where the
competitor charges $326). That is the "fat-finger ~10×" error the data notes warn about. No
prices are zero or negative in this cut.

**Therefore:** `clean()` drops rows where the price is non-positive or more than 3× away from
the competitor quote (1 row here). The 3× threshold is far outside the normal 0.8×–1.3× band, so it
only catches data errors, not genuinely expensive fares. With a squared-error loss, leaving
that one row in would let a single typo pull the whole fit.

In [ ]:
cdf = clean(df)
print("after clean():", cdf.shape)

## 5. Leakage risk

The prediction moment is **when the fare is quoted**. A feature is allowed only if it is known
at that moment.

| column | known at quote time? | decision |
|---|---|---|
| `days_to_departure`, `month`, `departure_hour`, weekend/holiday flags | yes, it describes the itinerary being priced | keep |
| `origin`, `destination`, `distance_miles`, `nonstop` | yes, fixed itinerary attributes | keep (encoded compactly, §6) |
| `cabin` | yes, the product being priced | keep |
| `load_factor` | yes **if** it is the load at quote time (the data notes describe it as seats already sold) | keep, flagged as an open PM question |
| `competitor_min_price` | yes, a market quote captured at search time, not derived from our price | keep, plus a sensitivity run without it |
| `base_fare`, `taxes_fees` | they sum to the target | not in this file; would be dropped |

The one real leakage trap in this data family (`base_fare + taxes_fees`) is not present.
The subtler risk is `competitor_min_price`: it explains a lot of the variance, and if in
production it were captured *after* AA's own price (competitors reacting to us), it would
leak. Its correlation with the target is shown below.

In [ ]:
num = cdf.select_dtypes("number")
corr = num.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print(corr.round(2).to_string())
print("log-log corr(competitor, price):",
      round(np.log(cdf["competitor_min_price"]).corr(np.log(cdf[TARGET])), 3))

**Therefore:** we keep `competitor_min_price` (on a log scale) but D4 also scores every
candidate **without** it. That way we know how much of the accuracy depends on the competitor
feed, and what the model is worth if the feed is unavailable or arrives late.

## 6. Relationships that bear on model choice

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.scatterplot(data=cdf, x="days_to_departure", y=TARGET, hue="cabin", hue_order=CABIN_ORDER,
                ax=axes[0, 0], s=20)
axes[0, 0].set(yscale="log", title="Advance-purchase curve")
axes[0, 0].axvline(14, color="k", ls="--", lw=1)
sns.scatterplot(data=cdf, x="distance_miles", y=TARGET, hue="cabin", hue_order=CABIN_ORDER,
                ax=axes[0, 1], s=20, legend=False)
axes[0, 1].set(xscale="log", yscale="log", title="Distance (log-log)")
sns.scatterplot(data=cdf, x="load_factor", y=TARGET, hue="cabin", hue_order=CABIN_ORDER,
                ax=axes[1, 0], s=20, legend=False)
axes[1, 0].set(yscale="log", title="Load factor")
sns.boxplot(data=cdf.assign(peak=np.where(cdf["is_weekend_departure"].astype(bool)
                                           | cdf["is_holiday_period"].astype(bool)
                                           | cdf["month"].isin([6, 7, 8]), "peak", "off-peak")),
            x="peak", y=TARGET, ax=axes[1, 1])
axes[1, 1].set(yscale="log", title="Peak (weekend/holiday/summer) vs off-peak")
fig.tight_layout()
fig.savefig(VISUALS / "eda_relationships.png", dpi=120)
plt.show()

In [ ]:
cdf = cdf.assign(dtd_bucket=pd.cut(cdf["days_to_departure"], [-1, 7, 14, 30, 60, 120],
                                   labels=["0-7", "8-14", "15-30", "31-60", "61-120"]))
med = cdf.groupby("dtd_bucket", observed=True)[TARGET].median()
rel = (cdf[TARGET] / cdf.groupby("cabin", observed=True)[TARGET].transform("median"))
print("median price by advance-purchase bucket:\n", med.round(0).to_string())
print("\nprice relative to cabin median, by bucket:\n",
      rel.groupby(cdf["dtd_bucket"], observed=True).median().round(2).to_string())

In [ ]:
card = pd.Series({
    "origin": cdf["origin"].nunique(),
    "destination": cdf["destination"].nunique(),
    "route (origin-destination)": (cdf["origin"].astype(str) + "-" + cdf["destination"].astype(str)).nunique(),
})
print(card.to_string())

What the plots and tables show:

- **Advance purchase is non-linear.** Relative to its cabin's median, a fare is ~0.95–1.07×
  anywhere beyond 14 days out, then jumps to ~1.8× inside 14 days. That is a step, not a slope. A straight line in
  `days_to_departure` cannot represent that, so we add a `last_minute` (≤ 14 days) flag.
- **Distance is roughly linear on a log-log scale.** So we use `log_distance`.
- **Cabin multiplies price** (§3). We use an ordered `cabin_rank` (0–3), which a log-target
  model can use directly and a tree can split on.
- **Load factor and the peak flags have weaker, noisier effects.** Kept, but they will not
  carry the model.
- **Airports have high cardinality:** ~24 origins and ~25 destinations, and almost every route
  appears only once or twice (154 distinct routes in 199 rows). One-hot encoding them would add ~50 sparse columns
  that memorise single rows. We replace them with `org_is_hub` / `des_is_hub`, and
  `log_distance` carries the route-length information.

**Therefore, model family: gradient-boosted trees, with a regularised linear (Ridge) model as
the baseline it has to beat.** The reasons, in the D2 order:

1. *Data shape:* a cross-section of quotes with no time index, so ARIMA/LSTM are disqualified.
2. *Loss:* the target is skewed and multiplicative, so we fit on log price (squared error on
   log ≈ relative error in dollars).
3. *Signal:* there are non-linear effects (the 14-day cliff) and likely interactions
   (cabin × advance purchase) that trees capture without hand-built terms.
4. *Cost/size:* with 200 rows a heavily parameterised model will overfit, so the final
   choice must be confirmed by cross-validation in D4. If Ridge on log price is as good,
   its readability wins.

## 7. The feature set

The feature logic lives in `features.py` (`clean()` + `build_features()`) so D4 and D5 import
it instead of copying it. Per-feature rationale: `results/feature-set.md`.

In [ ]:
X, y = build_features(cdf)
print("X:", X.shape, "| y:", y.shape)
assert list(X.columns) == FEATURES
assert X.notna().all().all(), "feature frame has gaps"
X.describe().T.round(2)